<a href="https://colab.research.google.com/github/akshita123454/mineral-exploration-cps/blob/main/ndvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.0 MB/s eta 0:00:00


In [1]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project = 'mineral-exploration-project')

In [ ]:

from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('elev_and_slope.csv')

In [ ]:
cols_to_keep = [
    'latitude',
    'longitude',
    'elevation',
    'slope',
    'country',
    'state',
    'ore',
    'gangue',
    'dep_type',
    'hrock_type',
    'arock_type',
    'ore_ctrl',
    'commod1'
]

df_clean = df[cols_to_keep].copy()

In [ ]:
df_clean.to_csv('elev_and_slope_clean.csv', index=False)

In [ ]:
from google.colab import files
files.download('elev_and_slope_clean.csv')

In [ ]:

from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv('elev_and_slope_clean.csv')

In [8]:
# See all unique countries in dataset

countries = df['country'].dropna().unique()

print(countries)

['United States' 'Argentina' 'Ghana' 'Oman' 'Angola' 'Namibia' 'Botswana'
 'Indonesia' 'China' 'Gabon' 'Kenya' 'Algeria' 'Burma'
 'Congo (Brazzaville)' 'Egypt' 'India' 'Korea, South' 'Iran' 'Israel'
 "Cote D'Ivoire" 'Morocco' 'Malaysia' 'Mozambique' 'Pakistan'
 'Philippines' 'Tanzania' 'Mali' 'Turkey' 'Peru' 'Bolivia' 'Mexico'
 'Canada' 'Costa Rica' 'Russia' 'Brazil' 'Kazakhstan' 'Germany' 'Slovenia'
 'Austria' 'France' 'United Kingdom' 'Spain' 'Tunisia' 'Zimbabwe'
 'South Africa' 'Australia' 'Azerbaijan' 'Sweden' 'Norway' 'Bulgaria'
 'Hungary' 'Western Sahara' 'Burkina Faso' 'Uganda' 'New Zealand'
 'Denmark' 'Somalia' 'Mauritania' 'Niger' 'Senegal' 'Sierra Leone' 'Togo'
 'Zambia' 'Cyprus' 'Iraq' 'Japan' 'Jordan' 'Taiwan' 'Thailand' 'Guinea'
 'Congo (Kinshasa)' 'Mongolia' 'Guatemala' 'Poland' 'Laos' 'Italy'
 'Guyana' 'Jamaica' 'Venezuela' 'Fiji' 'Suriname' 'Cameroon'
 'Solomon Islands' 'Ireland' 'Malawi' 'Afghanistan' 'Finland' 'Sri Lanka'
 'Chile' 'Bosnia and Herzegovina' 'Montenegro'

In [4]:
import pandas as pd

df = pd.read_csv('elev_and_slope_clean.csv')
df.head(5)

,latitude,longitude,elevation,slope,country,state,ore,gangue,dep_type,hrock_type,arock_type,ore_ctrl,commod1
0,55.05612,-132.14344,417.0,34.061240,United States,Alaska,"Chalcopyrite, Covellite, Pyrite","Quartz, Sericite",NaN,Schist,NaN,NaN,Copper
1,55.52751,-132.68514,773.0,29.553010,United States,Alaska,"Chalcopyrite, Pyrite","Calcite, Quartz, Siderite",NaN,Diabase,NaN,Vein Follows Contact,Copper
2,55.97751,-132.99906,58.0,3.311626,United States,Alaska,"Chalcopyrite, Pyrite, Sphalerite",Quartz,NaN,Siltstone,NaN,NaN,Copper
3,55.52195,-132.68653,653.0,20.563173,United States,Alaska,"Galena, Malachite, Pyrite",NaN,NaN,Granite,Granite,NaN,Gold
4,55.14556,-132.05233,36.0,8.804730,United States,Alaska,Pyrite,NaN,NaN,Mica Schist,NaN,NaN,Gold


In [12]:
nearby_countries = [
    'India',
    'Pakistan',
    'Bangladesh',
    'Nepal',
    'Sri Lanka',
    'Burma',   # Myanmar in your dataset
    'China'
]

df_region = df[df['country'].isin(nearby_countries)]

print(df_region.shape)
df_region.head()


(3582, 13)


,latitude,longitude,elevation,slope,country,state,ore,gangue,dep_type,hrock_type,arock_type,ore_ctrl,commod1
3639,18.61144,109.30263,227.0,5.878424,China,Liaoning,NaN,NaN,NaN,Dolomite,NaN,Mn-Bearing Searies,Manganese
3729,41.93420,93.00833,925.0,3.046159,China,Xinjiang* [Sinkiang],NaN,NaN,NaN,NaN,NaN,NaN,Manganese
3754,40.40652,108.40366,1242.0,9.575106,China,Nei Mongol* (Inner Mongolia),NaN,NaN,NaN,NaN,NaN,NaN,Manganese
3805,32.20980,114.50770,65.0,2.379133,China,Hunan,NaN,NaN,NaN,NaN,NaN,NaN,Manganese
3806,26.30470,118.51181,428.0,30.695847,China,Fujian [Fukien],NaN,NaN,NaN,NaN,NaN,NaN,Manganese


In [14]:
# Remove invalid rows
df_region = df_region.dropna(subset=['latitude', 'longitude'])

# Convert to numeric (in case of strings)
df_region['latitude'] = pd.to_numeric(df_region['latitude'], errors='coerce')
df_region['longitude'] = pd.to_numeric(df_region['longitude'], errors='coerce')

# Drop again after conversion
df_region = df_region.dropna(subset=['latitude', 'longitude'])

# Remove invalid ranges
df_region = df_region[(df_region['latitude'] >= -90) & (df_region['latitude'] <= 90)]
df_region = df_region[(df_region['longitude'] >= -180) & (df_region['longitude'] <= 180)]

df_region.reset_index(drop=True, inplace=True)

print("Cleaned dataset size:", len(df_region))

Cleaned dataset size: 3582


In [15]:
df_region.to_csv('elev_and_slope_india.csv', index=False)

In [16]:
from google.colab import files
files.download('elev_and_slope_india.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
from google.colab import files

uploaded = files.upload()

Saving elev_and_slope_india.csv to elev_and_slope_india (1).csv


In [50]:
import pandas as pd

df = pd.read_csv('elev_and_slope_india.csv')

In [51]:
sentinel = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

In [52]:
def get_ndvi_image(lat, lon):
    point = ee.Geometry.Point([lon, lat])

    image = (sentinel
             .filterBounds(point)
             .filterDate('2019-01-01', '2021-12-31')
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
             .median())

    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

    return ndvi.sample(point, 10).first().get('NDVI')

In [53]:
def df_to_ee(df):
    features = []
    for i in range(len(df)):
        lat = df.iloc[i]['latitude']
        lon = df.iloc[i]['longitude']

        point = ee.Geometry.Point([lon, lat])
        feature = ee.Feature(point, {
            'Latitude': lat,
            'Longitude': lon
        })

        features.append(feature)

    return ee.FeatureCollection(features)



In [54]:
image = (sentinel
         .filterDate('2019-01-01', '2021-12-31')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
         .median())

ndvi_image = image.normalizedDifference(['B8', 'B4']).rename('NDVI')

In [55]:
def get_ndvi_fc(fc):
    return ndvi_image.sampleRegions(
        collection=fc,
        scale=10,
        geometries=False   # 🔥 VERY IMPORTANT FIX
    )

In [56]:
import pandas as pd

def ee_to_df(fc):
    features = fc.getInfo()['features']

    data = []
    for f in features:
        props = f['properties']
        data.append(props)

    return pd.DataFrame(data)

In [57]:
batch_size = 1000

results = []

for i in range(0, len(df), batch_size):

    print(f"Processing batch {i} to {i+batch_size}")

    batch = df.iloc[i:i+batch_size]

    fc = df_to_ee(batch)

    ndvi_fc = get_ndvi_fc(fc)

    ndvi_df = geemap.ee_to_df(ndvi_fc)

    results.append(ndvi_df)

Processing batch 0 to 1000
Processing batch 1000 to 2000
Processing batch 2000 to 3000


Processing batch 3000 to 4000


In [58]:
ndvi_full = pd.concat(results)

ndvi_full.head()

,Latitude,Longitude,NDVI
0,18.61144,109.30263,0.684521
1,41.93420,93.00833,0.053832
2,40.40652,108.40366,0.155052
3,32.20980,114.50770,0.350995
4,26.30470,118.51181,0.784119


In [62]:
ndvi_full = ndvi_full.groupby(
    ['Latitude', 'Longitude'],
    as_index=False
)['NDVI'].mean()


In [64]:
final_df = df.merge(
    ndvi_full[['Latitude', 'Longitude', 'NDVI']],
    left_on=['latitude', 'longitude'],
    right_on=['Latitude', 'Longitude'],
    how='left'
)
final_df.head()
print(len(final_df))

3582


In [65]:
final_df.drop(columns=['Latitude', 'Longitude'], inplace=True)

In [67]:
final_df.to_csv('india_elev_slope_ndvi.csv', index=False)

In [68]:
from google.colab import files
files.download('india_elev_slope_ndvi.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>